# Optional · Error analysis

This exploration is not part of the required run sequence. Return to [01_run_model_day.ipynb](../01_run_model_day.ipynb) for controlled benchmark outputs.


# 09 — Error analysis

Analyze classification, localization, duplicates/background, misses, tiny-object misses, confidence calibration, occlusion/truncation, and density slices. Qualitative images must be selected by measured differences, not cherry-picked.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

SMOKE_TEST = os.environ.get("SMOKE_TEST", "0").lower() in {"1", "true", "yes", "on"}
try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IS_COLAB = False
REPOSITORY_URL = os.environ.get(
    "BENCHMARK_REPOSITORY_URL",
    "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git",
)
REPOSITORY_BRANCH = os.environ.get("BENCHMARK_REPOSITORY_BRANCH", "main")
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/aerial-object-detection-benchmark")
    if not (REPO_DIR / ".git").is_dir():
        subprocess.run(
            ["git", "clone", "--branch", REPOSITORY_BRANCH, REPOSITORY_URL, str(REPO_DIR)],
            check=True,
        )
else:
    REPO_DIR = Path(os.environ.get("BENCHMARK_REPO_ROOT", Path.cwd())).resolve()
if not (REPO_DIR / "pyproject.toml").is_file():
    raise RuntimeError(f"Repository root is invalid: {REPO_DIR}")
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if IS_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-dataset-colab.txt"],
        check=True,
    )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
DRIVE_ROOT = os.environ.get(
    "VISDRONE_DRIVE_ROOT",
    "/content/drive/MyDrive/visdrone_architecture_benchmark"
    if IS_COLAB
    else str(REPO_DIR / ".notebook-smoke"),
)
from src.paths import ProjectPaths
from src.reproducibility import seed_everything
from src.utils.environment import collect_environment
paths = ProjectPaths.from_value(DRIVE_ROOT).create()
seed_everything(42)
print({"repo": str(REPO_DIR), "storage": str(paths.root), "smoke_test": SMOKE_TEST})
collect_environment()


In [ ]:
from src.paths import ProjectPaths
from src.reproducibility import seed_everything
from src.utils.environment import collect_environment
paths = ProjectPaths.from_value(DRIVE_ROOT).create()
seed_everything(42)
collect_environment()

In [ ]:
import json, pandas as pd
files=sorted(paths.evaluation.glob("*__metrics.json"))
summary=[]
for f in files:
    d=json.loads(f.read_text()); summary.append({"model":d.get("model_id"),"mAP":d.get("mAP"),"APtiny":d.get("APtiny"),**d.get("error_counts",{})})
pd.DataFrame(summary)

## Automatic image selection

Use per-image TP/FP/FN and tiny-miss counts to select best, worst, largest pairwise differences, most tiny misses, and most false positives. Save a selection manifest before rendering side-by-side predictions.